## Governance — Grants, Row-Level Security, Column-Level Security
1. **Grants:** read-only access to the business gold tables for all workspace users 
2. **RLS:** users see only the categories allowed for them in `access_rules`.
3. **CLS:** the `dislikes` column is masked for users without permission.

In [0]:
%run ./00_config

### 1. Grants

In [0]:
reader_group = "account users"

spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.{gold_schema} TO `{reader_group}`")

for table in gold.values():
    spark.sql(f"GRANT SELECT ON TABLE {table} TO `{reader_group}`")

In [0]:
display(spark.sql(f"SHOW GRANTS ON TABLE {gold['fact_video_daily_stats']}"))

### 2. Access rules (entitlement table)


In [0]:
access_rules = f"{catalog}.{gold_schema}.access_rules"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {access_rules} (
        user_email           STRING  NOT NULL COMMENT 'User the rule applies to (matches current_user())',
        allowed_category_id  INT              COMMENT 'Category the user may see; NULL means all categories',
        can_see_dislikes     BOOLEAN NOT NULL COMMENT 'TRUE if the user may see the real dislikes values'
    )
    COMMENT 'Entitlement table for row filters and column masks on gold tables. Not granted to regular users.'
""")

me = spark.sql("SELECT current_user()").first()[0]

spark.sql(f"DELETE FROM {access_rules} WHERE user_email = '{me}'")
spark.sql(f"INSERT INTO {access_rules} VALUES ('{me}', NULL, TRUE)")

display(spark.table(access_rules))

In [0]:
mentor = "lbiel@softserve.academy"   

spark.sql(f"DELETE FROM {access_rules} WHERE user_email = '{mentor}'")
spark.sql(f"INSERT INTO {access_rules} VALUES ('{mentor}', NULL, TRUE)")

display(spark.table(access_rules))

### 3. Row-Level Security

In [0]:
rls_function = f"{catalog}.{gold_schema}.rls_category_filter"

spark.sql(f"""
    CREATE OR REPLACE FUNCTION {rls_function}(p_category_id INT)
    RETURNS BOOLEAN
    COMMENT 'Row filter: TRUE if current_user() may see rows of this category according to access_rules'
    RETURN EXISTS (
        SELECT 1
        FROM {access_rules} AS r
        WHERE r.user_email = current_user()
          AND (r.allowed_category_id IS NULL OR r.allowed_category_id = p_category_id)
    )
""")

In [0]:
for table in [gold["fact_video_daily_stats"], gold["agg_category_daily"]]:
    spark.sql(f"ALTER TABLE {table} SET ROW FILTER {rls_function} ON (category_id)")

In [0]:
rls_test_sql = f"""
    SELECT c.category_name, COUNT(*) AS rows_cnt
    FROM {gold['fact_video_daily_stats']} AS f
    JOIN {gold['dim_category']} AS c ON f.category_id = c.category_id
    GROUP BY c.category_name
    ORDER BY rows_cnt DESC
"""

display(spark.sql(rls_test_sql))

In [0]:
spark.sql(f"UPDATE {access_rules} SET allowed_category_id = 10 WHERE user_email = '{me}'")
display(spark.sql(rls_test_sql))

In [0]:
spark.sql(f"UPDATE {access_rules} SET allowed_category_id = NULL WHERE user_email = '{me}'")

display(spark.sql(rls_test_sql))

### 4. Column-Level Security

In [0]:
cls_function = f"{catalog}.{gold_schema}.cls_mask_dislikes"

spark.sql(f"""
    CREATE OR REPLACE FUNCTION {cls_function}(p_dislikes BIGINT)
    RETURNS BIGINT
    COMMENT 'Column mask: returns real dislikes only if current_user() has can_see_dislikes = TRUE in access_rules, otherwise NULL'
    RETURN CASE
        WHEN EXISTS (
            SELECT 1
            FROM {access_rules} AS r
            WHERE r.user_email = current_user()
              AND r.can_see_dislikes
        )
        THEN p_dislikes
        ELSE NULL
    END
""")

In [0]:
spark.sql(f"""
    ALTER TABLE {gold['fact_video_daily_stats']}
    ALTER COLUMN dislikes SET MASK {cls_function}
""")

In [0]:
cls_test_sql = f"""
    SELECT f.video_id, f.views, f.likes, f.dislikes
    FROM {gold['fact_video_daily_stats']} AS f
    ORDER BY f.views DESC
    LIMIT 5
"""

display(spark.sql(cls_test_sql))
display(spark.sql(f"SELECT SUM(dislikes) AS total_dislikes FROM {gold['fact_video_daily_stats']}"))

In [0]:
spark.sql(f"UPDATE {access_rules} SET can_see_dislikes = FALSE WHERE user_email = '{me}'")

display(spark.sql(cls_test_sql))
display(spark.sql(f"SELECT SUM(dislikes) AS total_dislikes FROM {gold['fact_video_daily_stats']}"))

In [0]:
spark.sql(f"UPDATE {access_rules} SET can_see_dislikes = TRUE WHERE user_email = '{me}'")

display(spark.sql(cls_test_sql))